In [1]:
# Cell 2: imports + models + Chroma client
import os, json
from sentence_transformers import SentenceTransformer
from transformers import pipeline

import chromadb
from chromadb.config import Settings

# Embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# QA pipeline (DistilBERT-SQuAD extractive QA)
qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad",
    tokenizer="distilbert-base-cased-distilled-squad"
)

# Persistent Chroma client
client = chromadb.PersistentClient(
    path="./chroma_db",
    settings=Settings(anonymized_telemetry=False)
)
    

/home/hazem/dev/Assist/glob_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cpu


In [2]:
# Cell 3: create or load your collection
collection = client.get_or_create_collection(name="my_study_docs")

In [3]:
# Cell 4: helper to load chunks from your JSON files
PREPROCESSED_FOLDER = "preprocessed"

def load_chunks_from_json(fp):
    with open(fp, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data.get("file_path", os.path.basename(fp)), data.get("chunks", [])


In [4]:
# Cell 5: embed & add chunks
def embed_and_add(source, chunks):
    embs = embed_model.encode(chunks, show_progress_bar=True)
    ids = [f"{source}_{i}" for i in range(len(chunks))]
    metas = [{"source_file": source, "idx": i} for i in range(len(chunks))]

    collection.add(
        ids=ids,
        embeddings=embs.tolist(),
        metadatas=metas,
        documents=chunks,
    )


In [5]:
# Cell 6: ingest all JSONs once
def ingest_all():
    for fn in os.listdir(PREPROCESSED_FOLDER):
        if not fn.endswith(".json"): continue
        path = os.path.join(PREPROCESSED_FOLDER, fn)
        src, chks = load_chunks_from_json(path)
        if not chks:
            print(f"  • skip {fn} (no chunks)")
            continue
        print(f"  • embedding {len(chks)} chunks from {src}")
        embed_and_add(src, chks)
    print("► all embeddings stored.")
        

In [ ]:
# Cell 7: initialize the LangChain retriever
from langchain.vectorstores import Chroma

ingest_all()

  • embedding 1 chunks from Responsible Machine Learning/Exercices/01_exercise_recap.pdf


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.10it/s]


  • embedding 1 chunks from Responsible Machine Learning/Exercices/01_-_Basics/02_homework.pdf


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.52it/s]


  • embedding 7 chunks from Deep Learning for Natural Language and Code/Lecture Notes/03_Tokenization.pdf


...

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.08it/s]


  • skip Zobel-Chapter3-Reading-And-Reviewing.pdf.json (no chunks)
  • embedding 1 chunks from Responsible Machine Learning/Exercices/01_-_Basics/ftest.txt


Batches: 100%|██████████| 1/1 [00:00<00:00, 40.13it/s]
/tmp/ipykernel_155079/4166879037.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vector_store = Chroma(collection=collection)


► all embeddings stored.


TypeError: Chroma.__init__() got an unexpected keyword argument 'collection'

In [13]:
# Cell 7: initialize the retriever with an embedding function
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma

# wrap your existing Chroma collection, supplying the embedder for queries
hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store = Chroma(
    collection_name="my_study_docs",
    embedding_function=hf_embeddings
)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})
# create a retriever that pulls top-3 docs


In [20]:
# Cell 8: define answer() with an empty‐hit guard
def answer(query: str) -> str:
    docs = retriever.get_relevant_documents(query)
    if not docs:
        return "⚠️ No relevant docs found. Did you run ingest_all()?"
    context = "\n\n".join(d.page_content for d in docs)
    print(f"---\nContext:\n{context}\n---\n")
    result = qa_pipeline(question=query, context=context)
    return result["answer"]


In [22]:
try:
    count = collection.count()  
    print(f"▶️ Embeddings in collection: {count}")
except:
    print("⚠️ collection.count() not available; try retrieving any ID list instead.")


▶️ Embeddings in collection: 1050


In [ ]:
# Cell 10: ingest and test
ingest_all()   # run your ingestion routine

  • embedding 1 chunks from Responsible Machine Learning/Exercices/01_exercise_recap.pdf


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.79it/s]


  • embedding 1 chunks from Responsible Machine Learning/Exercices/01_-_Basics/02_homework.pdf


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.28it/s]


  • embedding 7 chunks from Deep Learning for Natural Language and Code/Lecture Notes/03_Tokenization.pdf


...

  • embedding 5 chunks from Practical Parallel Programming/Lecture Notes/01-intro.pdf


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.42s/it]


  • skip Zobel-Chapter3-Reading-And-Reviewing.pdf.json (no chunks)
  • embedding 1 chunks from Responsible Machine Learning/Exercices/01_-_Basics/ftest.txt


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.08it/s]


► all embeddings stored.
⚠️ No relevant docs found. Did you run ingest_all()?


In [24]:
print(answer("What is a batch size?"))

⚠️ No relevant docs found. Did you run ingest_all()?
